# Ingesting

In [3]:
from pyspark.sql import SparkSession
from pathlib import Path

file_path = Path.cwd()
data_path = file_path.parent / "data" / "e-shop_clothing_2008.csv"

spark = SparkSession.builder.appName("clickstream-exploration").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv(
    str(data_path),
    header=True,
    inferSchema=True,
    sep=";"
)
df.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session ID: integer (nullable = true)
 |-- page 1 (main category): integer (nullable = true)
 |-- page 2 (clothing model): string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price 2: integer (nullable = true)
 |-- page: integer (nullable = true)



# Cleaning

## Removing odd characters

In [4]:
import re

for col in df.columns:
    new_name = re.sub(r'[ ()/]+', '_', col).strip('_').lower()
    df = df.withColumnRenamed(col, new_name)

df.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- page_1_main_category: integer (nullable = true)
 |-- page_2_clothing_model: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model_photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_2: integer (nullable = true)
 |-- page: integer (nullable = true)



## Merging day, month,year into DATE

In [5]:
from pyspark.sql import functions as F

df = df.withColumn(
    "date",
    F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.col("day")), "yyyy-M-d")
).drop("year", "month", "day")

In [6]:
df.head(1)

[Row(order=1, country=29, session_id=1, page_1_main_category=1, page_2_clothing_model='A13', colour=1, location=5, model_photography=1, price=28, price_2=2, page=1, date=datetime.date(2008, 4, 1))]

## Handling catagories

In [8]:
from pyspark.sql.types import StringType

In [9]:
df = df.withColumn("page_1_main_category", F.col("page_1_main_category").cast(StringType()))
df = df.withColumn("colour", F.col("colour").cast(StringType()))
df = df.withColumn("location", F.col("location").cast(StringType()))
df = df.withColumn("model_photography", F.col("model_photography").cast(StringType()))

In [10]:
df = df.withColumn("price_2", F.col("price_2") == 1)

## Checking for nulls

In [11]:
from pyspark.sql.functions import col, sum as spark_sum

df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-----+-------+----------+--------------------+---------------------+------+--------+-----------------+-----+-------+----+----+
|order|country|session_id|page_1_main_category|page_2_clothing_model|colour|location|model_photography|price|price_2|page|date|
+-----+-------+----------+--------------------+---------------------+------+--------+-----------------+-----+-------+----+----+
|    0|      0|         0|                   0|                    0|     0|       0|                0|    0|      0|   0|   0|
+-----+-------+----------+--------------------+---------------------+------+--------+-----------------+-----+-------+----+----+



## Checking for repeated *session_ids* or *orders*

In [12]:
df.groupBy("session_id", "order").count().filter(F.col("count") > 1).show()

+----------+-----+-----+
|session_id|order|count|
+----------+-----+-----+
+----------+-----+-----+



## Saving the cleaned df

In [14]:
output_path = Path.cwd().parent / "output" / "cleaned.parquet"
df.write.parquet(str(output_path), mode="overwrite")

# SESSIONizing